In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Topological Sorting

The function `topoSort` implements <em style="color:blue">Kahn's algorithm</em> for 
<em style="color:blue">topological sorting</em>.
- The first argument `T` is the set of the nodes of the directed graph.
- The second argument `D` is the set of the edges. Edges are represented as pairs.

The function returns a list `Sorted` that contains all nodes of $T$.  If
`Sorted[i] = x`, `Sorted[j] = y`, and `(x, y) in D`,
then we must have `i < j`.

In [ ]:
function topoSort(T: Set<number>, D: Array<[number, number]>): number[] {
    const Parents: Map<number, Set<number>> = new Map();
    const Children: Map<number, Set<number>> = new Map();
    for (const t of T) {
        Parents.set(t, new Set());
        Children.set(t, new Set());
    }
    for (const [s, t] of D) {
        Children.get(s)!.add(t);
        Parents.get(t)!.add(s);
    }
    const Orphans: Set<number> = new Set();
    for (const [t, P] of Parents.entries()) {
        if (P.size === 0) {
            Orphans.add(t);
        }
    }
    const Sorted: number[] = [];
    let count = 0;
    const TCopy = new Set(T);
    while (TCopy.size > 0) {
        if (Orphans.size === 0) {
            throw new Error('The graph is cyclic!');
        }
        const orphansIterator = Orphans.values();
        const t = orphansIterator.next().value;
        count += 1;
        Orphans.delete(t);
        TCopy.delete(t);
        Sorted.push(t);
        for (const s of Children.get(t)!) {
            Parents.get(s)!.delete(t);
            if (Parents.get(s)!.size === 0) {
                Orphans.add(s);
            }
        }
    }
    return Sorted;
}

# Graphical Representation

The function `toDot(Edges, Order)` takes two arguments:
- `Edges` is a set of pairs of the form `(x, y)` where `x` and `y` are nodes of a graph
  and `(x, y)` is a directed edge from `x`to `y`.
- `Order` is a dictionary assigning natural numbers to some of the nodes.

The set of edges is displayed as a directed graph and for those nodes `x` such that `Order[x]` is defined, both `x` and the label `Order[x]` is depicted.

In [ ]:
import { Graphviz } from "@hpcc-js/wasm";
import { display } from "tslab";

async function toDot(Edges: Array<[number, number]>, Order: Map<number, number> | null): Promise<void> {
    const V = new Set<number>();
    for (const [x, y] of Edges) {
        V.add(x);
        V.add(y);
    }
    let dot = 'digraph {\n';
    dot += 'rankdir=LR;\n';
    dot += 'size="8,5";\n';
    dot += 'node [shape=record style=rounded];\n\n';
    for (const x of V) {
        if (Order !== null && Order.has(x)) {
            const o = Order.get(x)!;
            dot += `${x} [label="{${x}|${o}}"];\n`;
        } else {
            dot += `${x};\n`;
        }
    }
    dot += '\n';
    for (const [u, v] of Edges) {
        dot += `${u} -> ${v};\n`;
    }
    dot += '}\n';
    const gv = await Graphviz.load();
    const svg = gv.layout(dot, "svg", "dot");
    display.html(svg);
}

The version of `topoSort` that is given below adds some animation to the algorithm.

In [ ]:
async function topoSort(T: Set<number>, D: Array<[number, number]>): Promise<number[]> {
    console.log('_'.repeat(100));
    await toDot(D, null);
    const Parents: Map<number, Set<number>> = new Map(); 
    const Children: Map<number, Set<number>> = new Map();
    for (const t of T) {
        Parents.set(t, new Set());
        Children.set(t, new Set());
    }
    for (const [s, t] of D) {
        Children.get(s)!.add(t);
        Parents.get(t)!.add(s);
    }
    const Orphans: Set<number> = new Set();
    for (const [t, P] of Parents.entries()) {
        if (P.size === 0) {
            Orphans.add(t);
        }
    }
    const Sorted: number[] = [];
    let count = 0;
    const Order: Map<number, number> = new Map();
    const TCopy = new Set(T);
    while (TCopy.size > 0) {
        if (Orphans.size === 0) {
            throw new Error('The graph is cyclic!');
        }
        const orphansIterator = Orphans.values();
        const t = orphansIterator.next().value;
        Order.set(t, count);
        count += 1;
        Orphans.delete(t);
        TCopy.delete(t);
        Sorted.push(t);
        for (const s of Children.get(t)!) {
            Parents.get(s)!.delete(t);
            if (Parents.get(s)!.size === 0) {
                Orphans.add(s);
            }
        }
        console.log('_'.repeat(80));
        await toDot(D, Order);
    }
    return Sorted;
}

# Testing

In [ ]:
async function demo() {
    const T = new Set([5, 7, 3, 11, 8, 2, 9, 10]);
    const D: Array<[number, number]> = [
        [5, 11], [7, 11], [7, 8], [3, 8], [3, 10],
        [11, 2], [11, 9], [11, 10], [8, 9]
    ];
    const S = await topoSort(T, D);
    console.log(S);
}

In [ ]:
await demo();